<a href="https://colab.research.google.com/github/ivasylenko1/research_seminar_cv_search_project/blob/main/CV_embedding_finetune_golden.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning ембеддінгів на `FINAL_training_data.csv` (golden dataset)

Це вже НЕ auto-eval holdout — файл містить реальні релевантність-мітки
(`Relevance_Score`, судячи зі скріна шкала 0-3) для пар (job, candidate),
разом із джерелом (`Source`: BM25 / Dense), готовим `cv_text` та `job_text`.

Порівняно з попереднім пайплайном (Section 9 з auto-labeling) це велике
покращення: eval тепер буде проти справжніх міток, а не проти того самого
Qwen3-Reranker, який генерував training data.

**Що робить цей ноутбук:**
1. Завантажує `FINAL_training_data.csv` — весь файл іде на побудову training pairs
2. З нього будує positive/hard-negative пари
3. Окремо завантажує `Golden_Eval/eval_pool_*.csv` (кожен файл — готовий пул
   кандидатів для однієї вакансії) — це незалежний held-out eval-набір,
   який не перетинається з training data
4. Тренує Qwen3-Embedding з MultipleNegativesRankingLoss + Matryoshka
5. Верифікує збережену модель і порівнює baseline vs fine-tuned NDCG@10

### 1. Setup

In [1]:
# --- Setup ---

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # до import torch

import gc
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator

from google.colab import drive
drive.mount("/content/drive")

CONFIG = {
    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
    "output_dir": "/content/drive/MyDrive/qwen-cv-retriever-finetuned-golden",
    "golden_csv_path": "/content/drive/MyDrive/CV_rank_Datasets/FINAL_training_data.csv",
    "golden_eval_dir": "/content/drive/MyDrive/CV_rank_Datasets/Golden_Eval",
}
DOC_PROMPT = "Represent this candidate CV for job matching retrieval"
QUERY_PROMPT = "Find candidates matching this job vacancy"

gc.collect()
torch.cuda.empty_cache()

embed_model = SentenceTransformer(CONFIG["embedding_model"])  # без torch_dtype=float16 —
# конфліктує з bf16-тренуванням нижче (fp16 ваги + bf16/fp16 AMP разом ламають GradScaler)
embed_model.max_seq_length = 512
print(f"Модель завантажена: {CONFIG['embedding_model']}, max_seq_length={embed_model.max_seq_length}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'немає'}")


/tmp/ipykernel_635/4190533415.py:11: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_635/4190533415.py:17: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Модель завантажена: Qwen/Qwen3-Embedding-0.6B, max_seq_length=512
GPU: NVIDIA A100-SXM4-40GB


### 2. Завантаження golden dataset + автовизначення колонок

Автовизначення на випадок, якщо реальні назви колонок трохи відрізняються
від того, що видно на скріні (там частина заголовків обрізана).

In [2]:
# --- Завантаження CSV + автовизначення колонок ---

golden_df = pd.read_csv(CONFIG["golden_csv_path"])
print(f"Завантажено {len(golden_df)} рядків")
print("Колонки:", list(golden_df.columns))

def find_column(df, keywords):
    for col in df.columns:
        if any(kw in col.lower() for kw in keywords):
            return col
    raise ValueError(f"Не знайдено колонку за ключовими словами {keywords} серед {list(df.columns)}")

JOB_ID_COL = find_column(golden_df, ["job_id"])
CAND_ID_COL = find_column(golden_df, ["candidate_id"])
RELEVANCE_COL = find_column(golden_df, ["relevance"])
JOB_TEXT_COL = find_column(golden_df, ["job_text"])
CV_TEXT_COL = find_column(golden_df, ["cv_text"])
SOURCE_COL = find_column(golden_df, ["source"]) if any("source" in c.lower() for c in golden_df.columns) else None

print(f"\njob_id -> {JOB_ID_COL!r}, candidate_id -> {CAND_ID_COL!r}, relevance -> {RELEVANCE_COL!r}")
print(f"job_text -> {JOB_TEXT_COL!r}, cv_text -> {CV_TEXT_COL!r}, source -> {SOURCE_COL!r}")

print(f"\nУнікальних вакансій: {golden_df[JOB_ID_COL].nunique()}")
print(f"Унікальних кандидатів: {golden_df[CAND_ID_COL].nunique()}")
print(f"\nРозподіл Relevance_Score:\n{golden_df[RELEVANCE_COL].value_counts().sort_index()}")
if SOURCE_COL:
    print(f"\nРозподіл Source:\n{golden_df[SOURCE_COL].value_counts()}")


Завантажено 3163 рядків
Колонки: ['job_id', 'Candidate_ID', 'Relevance_Score_0_to_3', 'Source', 'Dense_Score', 'BM25_Score', 'full_name', 'title', 'seniority', 'location', 'candidate_id', 'skills', 'languages', 'cv_text', 'job_text']

job_id -> 'job_id', candidate_id -> 'Candidate_ID', relevance -> 'Relevance_Score_0_to_3'
job_text -> 'job_text', cv_text -> 'cv_text', source -> 'Source'

Унікальних вакансій: 52
Унікальних кандидатів: 1790

Розподіл Relevance_Score:
Relevance_Score_0_to_3
0     964
1    1293
2     714
3     192
Name: count, dtype: int64

Розподіл Source:
Source
Dense              1504
BM25               1303
Random Baseline     192
Dense + BM25        164
Name: count, dtype: int64


### 3. Дедублікація

Якщо один кандидат потрапив у пул вакансії і через BM25, і через Dense —
буде два рядки з однаковим (job_id, candidate_id). Прибираємо дублікати,
щоб не рахувати одну пару кандидат-вакансія двічі.

In [3]:
# --- Дедублікація за (job_id, candidate_id) ---

before = len(golden_df)
golden_df = golden_df.drop_duplicates(subset=[JOB_ID_COL, CAND_ID_COL], keep="first").reset_index(drop=True)
print(f"Рядків до дедублікації: {before}, після: {len(golden_df)}")

# Прибираємо рядки з порожнім текстом (про всяк випадок)
golden_df = golden_df.dropna(subset=[JOB_TEXT_COL, CV_TEXT_COL]).reset_index(drop=True)
print(f"Рядків з валідним текстом: {len(golden_df)}")


Рядків до дедублікації: 3163, після: 2662
Рядків з валідним текстом: 2662


### 4. Весь `FINAL_training_data.csv` йде в тренування

Без train/eval розділення — тут `golden_df` цілком використовується для
побудови training pairs. Окрема оцінка (Section 7) буде на незалежних
файлах з папки `Golden_Eval`, які не перетинаються з цим датасетом.

In [4]:
# --- Весь golden_df йде в тренування ---

train_df = golden_df
print(f"Рядків для тренування: {len(train_df)}, вакансій: {train_df[JOB_ID_COL].nunique()}")


Рядків для тренування: 2662, вакансій: 52


### 5. Positive / hard-negative пари

- **positive** = `Relevance_Score >= 2` (good match / perfect match, за шкалою
  вашого golden set 0-3)
- **hard negative** = `Relevance_Score <= 1` (not relevant / partial match) —
  кандидат був у пулі (тобто BM25 чи Dense вважали його схожим), але
  насправді не підходить. Саме такі приклади вчать модель тонкої
  дискримінації, а не випадкові негативи.

In [5]:
# --- Побудова positive/hard-negative пар ---

POSITIVE_THRESHOLD = 2   # Relevance_Score >= 2 -> positive
NEGATIVE_THRESHOLD = 1   # Relevance_Score <= 1 -> hard negative
MAX_NEG_PER_POS = 3       # обмеження, щоб не роздувати датасет однією вакансією з великою кількістю негативів

rows = []
for job_id, group in train_df.groupby(JOB_ID_COL):
    job_text = group[JOB_TEXT_COL].iloc[0]
    positives = group[group[RELEVANCE_COL] >= POSITIVE_THRESHOLD]
    negatives = group[group[RELEVANCE_COL] <= NEGATIVE_THRESHOLD]

    if len(positives) == 0 or len(negatives) == 0:
        continue  # немає з чого зробити пару для цієї вакансії

    neg_texts = negatives[CV_TEXT_COL].tolist()[:MAX_NEG_PER_POS]
    for _, pos_row in positives.iterrows():
        for neg_text in neg_texts:
            rows.append({
                "vacancy_text": job_text,
                "cv_text_positive": pos_row[CV_TEXT_COL],
                "cv_text_hard_negative": neg_text,
            })

train_pairs_df = pd.DataFrame(rows)
print(f"Training pairs: {len(train_pairs_df)}")
print(f"Вакансій з корисними парами: {train_pairs_df['vacancy_text'].nunique() if len(train_pairs_df) else 0}/{train_df[JOB_ID_COL].nunique()}")

assert len(train_pairs_df) > 0, "Не вдалось побудувати жодної пари — перевір пороги/дані"


Training pairs: 1215
Вакансій з корисними парами: 52/52


### 6. Training dataset для sentence-transformers

In [6]:
# --- Dataset для тренування ---

anchors = (QUERY_PROMPT + ": " + train_pairs_df["vacancy_text"]).tolist()
positives = (DOC_PROMPT + ": " + train_pairs_df["cv_text_positive"]).tolist()
negatives = (DOC_PROMPT + ": " + train_pairs_df["cv_text_hard_negative"]).tolist()

train_dataset = Dataset.from_dict({"anchor": anchors, "positive": positives, "negative": negatives})
print(f"train_dataset готовий: {len(train_dataset)} прикладів")


train_dataset готовий: 1215 прикладів


### 7. Eval data з окремих файлів `Golden_Eval/eval_pool_*.csv`

Кожен файл у цій папці — це вже готовий пул кандидатів для однієї вакансії
(структура колонок та сама, що й у `FINAL_training_data.csv`). Читаємо всі
файли з папки, тег кожного рядка джерельним файлом (про запас, якщо
`job_id` десь повториться між файлами) і будуємо `eval_data`.

In [7]:
# --- Завантаження всіх eval_pool_*.csv з папки Golden_Eval ---

import glob

eval_files = sorted(glob.glob(os.path.join(CONFIG["golden_eval_dir"], "*.csv")))
print(f"Знайдено {len(eval_files)} файлів у {CONFIG['golden_eval_dir']}:")
for f in eval_files:
    print(" ", os.path.basename(f))

assert len(eval_files) > 0, "Жодного CSV не знайдено — перевір шлях golden_eval_dir"

eval_parts = []
for f in eval_files:
    part = pd.read_csv(f)
    part["_source_file"] = os.path.basename(f)
    eval_parts.append(part)

eval_df = pd.concat(eval_parts, ignore_index=True)
eval_df = eval_df.dropna(subset=[JOB_TEXT_COL, CV_TEXT_COL]).reset_index(drop=True)
# групуємо за (job_id, файл) на випадок, якщо job_id повториться між різними вакансіями-файлами
eval_df["_group_key"] = eval_df[JOB_ID_COL].astype(str) + "__" + eval_df["_source_file"]

print(f"\nВсього рядків eval: {len(eval_df)}, унікальних вакансій: {eval_df['_group_key'].nunique()}")


Знайдено 10 файлів у /content/drive/MyDrive/CV_rank_Datasets/Golden_Eval:
  eval_pool_Cloud_Network_Engineer_Final.csv
  eval_pool_Data_Science_Engineer_Final.csv
  eval_pool_Fullstack_Python_LLM_Developer_Final.csv
  eval_pool_Head_of_Parser__Python___Web_S_Final.csv
  eval_pool_Intern_QA_Game_Engineer_Final.csv
  eval_pool_Junior_Java_Developer_Final.csv
  eval_pool_Middle_Embedded_Software_Engin_Final.csv
  eval_pool_Senior_Full_Stack_Developer__P_Final.csv
  eval_pool_Senior_Go_Engineer___Tech_Lead_Final.csv
  eval_pool_Strong_Junior_Node_js_Develope_Final.csv

Всього рядків eval: 629, унікальних вакансій: 10


In [8]:
# --- eval_data з Golden_Eval ---

eval_data = {}
for group_key, group in eval_df.groupby("_group_key"):
    label = group["_source_file"].iloc[0].replace(".csv", "")
    eval_data[label] = {
        "vacancy_text": group[JOB_TEXT_COL].iloc[0],
        "cv_texts": group[CV_TEXT_COL].tolist(),
        "labels": group[RELEVANCE_COL].tolist(),
    }

print(f"eval_data: {len(eval_data)} вакансій")
print(f"Кандидатів у eval, всього: {sum(len(d['cv_texts']) for d in eval_data.values())}")


eval_data: 10 вакансій
Кандидатів у eval, всього: 629


### 8. Evaluator

In [9]:
# --- IR Evaluator на реальних мітках ---

def build_ir_evaluator(eval_data, positive_threshold):
    queries, corpus, relevant_docs = {}, {}, {}
    for title, data in eval_data.items():
        qid = title
        queries[qid] = QUERY_PROMPT + ": " + data["vacancy_text"]
        relevant_docs[qid] = set()
        for i, (cv_text, label) in enumerate(zip(data["cv_texts"], data["labels"])):
            cid = f"{title}__{i}"
            corpus[cid] = DOC_PROMPT + ": " + cv_text
            if label >= positive_threshold:
                relevant_docs[qid].add(cid)
    return InformationRetrievalEvaluator(
        queries=queries, corpus=corpus, relevant_docs=relevant_docs,
        name="cv-vacancy-golden-eval",
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10], ndcg_at_k=[10],
        batch_size=8,
    )

evaluator = build_ir_evaluator(eval_data, POSITIVE_THRESHOLD)
print(f"Evaluator готовий: {len(evaluator.queries)} вакансій, {len(evaluator.corpus)} унікальних CV")


Evaluator готовий: 10 вакансій, 629 унікальних CV


### 9. Loss

In [10]:
# --- Loss: MultipleNegativesRankingLoss + Matryoshka ---

USE_MATRYOSHKA = True
MATRYOSHKA_DIMS = [1024, 768, 512, 256, 128]

base_loss = losses.MultipleNegativesRankingLoss(embed_model)
train_loss = losses.MatryoshkaLoss(embed_model, base_loss, matryoshka_dims=MATRYOSHKA_DIMS) if USE_MATRYOSHKA else base_loss
print("Loss готовий")


Loss готовий


### 10. Тренування (A100, bf16)

Налаштування враховують усі попередні виправлення:
- `bf16=True` / `fp16=False` — щоб не ловити конфлікт з `GradScaler`
- батч 8 + gradient accumulation 4 (ефективний батч 32) + gradient checkpointing —
  щоб не впасти в OOM
- `metric_for_best_model` вказано явно під назву метрики нашого evaluator'а
  (кастомний evaluator не рахує `eval_loss`, тому дефолт зламав би
  `load_best_model_at_end`)

In [11]:
# --- Тренування ---

args = SentenceTransformerTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    bf16=True,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="cv-vacancy-golden-eval_cosine_ndcg@10",
    greater_is_better=True,
    dataloader_num_workers=2,
)

trainer = SentenceTransformerTrainer(
    model=embed_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

trainer.train()

embed_model.save(CONFIG["output_dir"])
print(f"Модель збережена -> {CONFIG['output_dir']}")


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Cv-vacancy-golden-eval Cosine Accuracy@1,Cv-vacancy-golden-eval Cosine Accuracy@3,Cv-vacancy-golden-eval Cosine Accuracy@5,Cv-vacancy-golden-eval Cosine Accuracy@10,Cv-vacancy-golden-eval Cosine Precision@1,Cv-vacancy-golden-eval Cosine Precision@3,Cv-vacancy-golden-eval Cosine Precision@5,Cv-vacancy-golden-eval Cosine Precision@10,Cv-vacancy-golden-eval Cosine Recall@1,Cv-vacancy-golden-eval Cosine Recall@3,Cv-vacancy-golden-eval Cosine Recall@5,Cv-vacancy-golden-eval Cosine Recall@10,Cv-vacancy-golden-eval Cosine Ndcg@10,Cv-vacancy-golden-eval Cosine Mrr@10,Cv-vacancy-golden-eval Cosine Map@100
1,5.683105,No log,0.500000,0.700000,0.700000,0.700000,0.500000,0.400000,0.360000,0.320000,0.044354,0.103882,0.153410,0.275674,0.349571,0.583333,0.344759
2,2.277124,No log,0.500000,0.700000,0.700000,0.700000,0.500000,0.366667,0.380000,0.300000,0.041146,0.098619,0.170027,0.270527,0.337539,0.583333,0.338839
3,1.672395,No log,0.400000,0.700000,0.700000,0.700000,0.400000,0.400000,0.380000,0.310000,0.035263,0.103882,0.170027,0.272292,0.334081,0.533333,0.330161


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель збережена -> /content/drive/MyDrive/qwen-cv-retriever-finetuned-golden


### 11. Верифікація: модель дійсно видає ембеддінги?

Перевіряємо одразу, у свіжій змінній з диска — не довіряємо живому об'єкту
в пам'яті.

In [12]:
# --- Верифікація збереженої моделі ---

del trainer
del embed_model
gc.collect()
torch.cuda.empty_cache()

reloaded_model = SentenceTransformer(CONFIG["output_dir"])
test_vec = reloaded_model.encode("test sentence", normalize_embeddings=True)

print(f"Форма: {test_vec.shape}")
print(f"Норма (очікується ~1.0): {np.linalg.norm(test_vec):.4f}")
print(f"NaN у векторі: {np.isnan(test_vec).any()}")

assert test_vec.shape[0] > 0, "Порожній вектор — модель не зберегла pooling/dense шар!"
assert not np.isnan(test_vec).any(), "NaN у векторі — щось зламалось під час тренування!"
print("\n✅ Модель успішно видає ембеддінги очікуваної форми")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Форма: (1024,)
Норма (очікується ~1.0): 1.0034
NaN у векторі: False

✅ Модель успішно видає ембеддінги очікуваної форми


### 12. Порівняння baseline vs fine-tuned на реальних golden-мітках

In [13]:
# --- Порівняння NDCG@10 ---

from sklearn.metrics import ndcg_score

def compare_embeddings_ndcg(model, eval_data, k=10):
    rows = []
    for title, data in eval_data.items():
        query_vec = model.encode(data["vacancy_text"], prompt=QUERY_PROMPT, normalize_embeddings=True, batch_size=4)
        doc_vecs = model.encode(data["cv_texts"], prompt=DOC_PROMPT, normalize_embeddings=True, batch_size=4, show_progress_bar=False)
        scores = doc_vecs @ query_vec
        ndcg = ndcg_score(np.array([data["labels"]]), np.array([scores]), k=k)
        rows.append({"vacancy": str(title)[:40], f"NDCG@{k}": round(ndcg, 4)})
    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    print(f"Середній NDCG@{k}: {result_df[f'NDCG@{k}'].mean():.4f}")
    return result_df

print("=== Baseline (оригінальна Qwen3-Embedding) ===")
baseline_model = SentenceTransformer(CONFIG["embedding_model"])
baseline_results = compare_embeddings_ndcg(baseline_model, eval_data, k=10)

baseline_model.to("cpu")
del baseline_model
gc.collect()
torch.cuda.empty_cache()

print("\n=== Fine-tuned (на golden dataset) ===")
finetuned_results = compare_embeddings_ndcg(reloaded_model, eval_data, k=10)

improvement = finetuned_results["NDCG@10"].mean() - baseline_results["NDCG@10"].mean()
print(f"\nΔ NDCG@10: {improvement:+.4f}")


=== Baseline (оригінальна Qwen3-Embedding) ===


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

                                 vacancy  NDCG@10
eval_pool_Fullstack_Python_LLM_Developer   0.7850
   eval_pool_Data_Science_Engineer_Final   0.6081
 eval_pool_Intern_QA_Game_Engineer_Final   0.4445
eval_pool_Senior_Go_Engineer___Tech_Lead   0.6707
eval_pool_Strong_Junior_Node_js_Develope   0.6479
eval_pool_Senior_Full_Stack_Developer__P   0.3258
   eval_pool_Junior_Java_Developer_Final   0.2029
eval_pool_Head_of_Parser__Python___Web_S   0.3770
eval_pool_Middle_Embedded_Software_Engin   0.6159
  eval_pool_Cloud_Network_Engineer_Final   0.7727
Середній NDCG@10: 0.5451

=== Fine-tuned (на golden dataset) ===
                                 vacancy  NDCG@10
eval_pool_Fullstack_Python_LLM_Developer   0.6643
   eval_pool_Data_Science_Engineer_Final   0.4900
 eval_pool_Intern_QA_Game_Engineer_Final   0.3982
eval_pool_Senior_Go_Engineer___Tech_Lead   0.6610
eval_pool_Strong_Junior_Node_js_Develope   0.6354
eval_pool_Senior_Full_Stack_Developer__P   0.7909
   eval_pool_Junior_Java_Developer_

In [14]:
# --- Порівняння NDCG@10, Recall@10, MAP@10 ---

from sklearn.metrics import ndcg_score

def recall_at_k(labels, scores, k, positive_threshold):
    order = np.argsort(scores)[::-1][:k]
    labels = np.array(labels)
    n_relevant_total = (labels >= positive_threshold).sum()
    if n_relevant_total == 0:
        return None  # немає жодного релевантного кандидата для цієї вакансії — пропускаємо
    n_relevant_retrieved = (labels[order] >= positive_threshold).sum()
    return n_relevant_retrieved / n_relevant_total

def average_precision_at_k(labels, scores, k, positive_threshold):
    order = np.argsort(scores)[::-1][:k]
    labels = np.array(labels)
    relevant_mask = (labels[order] >= positive_threshold).astype(int)
    if relevant_mask.sum() == 0:
        return None  # жодного релевантного серед top-k — AP не визначений, пропускаємо
    precisions = np.cumsum(relevant_mask) / (np.arange(len(relevant_mask)) + 1)
    ap = (precisions * relevant_mask).sum() / relevant_mask.sum()
    return ap

def compare_embeddings_ndcg(model, eval_data, k=10, positive_threshold=POSITIVE_THRESHOLD):
    rows = []
    for title, data in eval_data.items():
        query_vec = model.encode(data["vacancy_text"], prompt=QUERY_PROMPT, normalize_embeddings=True, batch_size=4)
        doc_vecs = model.encode(data["cv_texts"], prompt=DOC_PROMPT, normalize_embeddings=True, batch_size=4, show_progress_bar=False)
        scores = doc_vecs @ query_vec
        labels = data["labels"]

        ndcg = ndcg_score(np.array([labels]), np.array([scores]), k=k)
        recall = recall_at_k(labels, scores, k, positive_threshold)
        ap = average_precision_at_k(labels, scores, k, positive_threshold)

        rows.append({
            "vacancy": str(title)[:40],
            f"NDCG@{k}": round(ndcg, 4),
            f"Recall@{k}": round(recall, 4) if recall is not None else None,
            f"AP@{k}": round(ap, 4) if ap is not None else None,
        })

    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    print(f"\nСередній NDCG@{k}: {result_df[f'NDCG@{k}'].mean():.4f}")
    print(f"Середній Recall@{k}: {result_df[f'Recall@{k}'].mean():.4f}  (по {result_df[f'Recall@{k}'].notna().sum()}/{len(result_df)} вакансіях, де є релевантні кандидати)")
    print(f"MAP@{k} (mean of AP@{k}): {result_df[f'AP@{k}'].mean():.4f}  (по {result_df[f'AP@{k}'].notna().sum()}/{len(result_df)} вакансіях)")
    return result_df

print("=== Baseline (оригінальна Qwen3-Embedding) ===")
baseline_model = SentenceTransformer(CONFIG["embedding_model"])
baseline_results = compare_embeddings_ndcg(baseline_model, eval_data, k=10)

baseline_model.to("cpu")
del baseline_model
gc.collect()
torch.cuda.empty_cache()

print("\n=== Fine-tuned (на golden dataset) ===")
finetuned_results = compare_embeddings_ndcg(reloaded_model, eval_data, k=10)

print("\n=== Δ (fine-tuned - baseline) ===")
print(f"Δ NDCG@10:   {finetuned_results['NDCG@10'].mean() - baseline_results['NDCG@10'].mean():+.4f}")
print(f"Δ Recall@10: {finetuned_results['Recall@10'].mean() - baseline_results['Recall@10'].mean():+.4f}")
print(f"Δ MAP@10:    {finetuned_results['AP@10'].mean() - baseline_results['AP@10'].mean():+.4f}")

=== Baseline (оригінальна Qwen3-Embedding) ===


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

                                 vacancy  NDCG@10  Recall@10  AP@10
eval_pool_Fullstack_Python_LLM_Developer   0.7850     0.3684 0.8598
   eval_pool_Data_Science_Engineer_Final   0.6081     0.0000    NaN
 eval_pool_Intern_QA_Game_Engineer_Final   0.4445     0.0000    NaN
eval_pool_Senior_Go_Engineer___Tech_Lead   0.6707     0.3529 0.7345
eval_pool_Strong_Junior_Node_js_Develope   0.6479     0.2500 0.6250
eval_pool_Senior_Full_Stack_Developer__P   0.3258     0.1000 0.1111
   eval_pool_Junior_Java_Developer_Final   0.2029     0.1000 0.3333
eval_pool_Head_of_Parser__Python___Web_S   0.3770     0.0000    NaN
eval_pool_Middle_Embedded_Software_Engin   0.6159     0.5000 0.4024
  eval_pool_Cloud_Network_Engineer_Final   0.7727     0.4545 0.6200

Середній NDCG@10: 0.5451
Середній Recall@10: 0.2126  (по 10/10 вакансіях, де є релевантні кандидати)
MAP@10 (mean of AP@10): 0.5266  (по 7/10 вакансіях)

=== Fine-tuned (на golden dataset) ===
                                 vacancy  NDCG@10  Recall@

---
### Примітки

- Це перший запуск, де eval іде проти справжніх golden-міток, а не проти
  auto-labels того самого Qwen3-Reranker — тому цифри тут надійніші за
  попередній прогін.
- Якщо `train_pairs_df` виявився замалим (мало вакансій із хоч одним
  positive І хоч одним hard negative) — знизь `POSITIVE_THRESHOLD` до 1
  або збільш `MAX_NEG_PER_POS`.
- Не бери модель з `output_dir/checkpoint-XXX` для інференсу — бери сам
  `CONFIG["output_dir"]`, куди `embed_model.save(...)` пише фінальну версію.